In [1]:
import pandas as pd
from tqdm import tqdm
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
import os
from multiprocessing import Pool
from scipy.stats import entropy

In [2]:
accessions=['at6923', 'at6929', 'at7143', 'at8285', 'at9104', 'at9336', 'at9503', 'at9578', 'at9744', 'at9762', 'at9806', 'at9830', 'at9847', 'at9852', 'at9879', 'at9883', 'at9900']

In [3]:
import warnings
warnings.simplefilter(action='ignore', category=pd.errors.PerformanceWarning)

# load sqanti classification file
at=pd.read_csv(f"/tmp/global2/lvaness/dl20-annex/final/isoseq/sqanti/classifications/at9900-hodgepodgemerged_classification.txt", sep="\t")

In [50]:
# get associated hodgepodge gene (one per row)
span = 2
at["hodgepodge_gene"]=at["associated_gene"].transform(lambda x: ["_".join(x.split("_")[i:i+span]) for i in range(0, len(x.split("_")), span)])
at_hodge=at.explode("hodgepodge_gene")
at_hodge=at_hodge.loc[at_hodge["hodgepodge_gene"].str.split("_").str[0]=="at9900"]

In [62]:
# start analysis per gene
possible_5degradation=["alternative_5end","5prime_fragment"]
genes_wo_deg=pd.DataFrame()
genes_uk=pd.DataFrame()
for gene in tqdm(at_hodge.groupby(by="hodgepodge_gene", dropna=False)):
    # handle genes with reference match an 5' alteration
    if 'reference_match' in gene[1]["subcategory"].tolist() and any(x in possible_5degradation for x in gene[1]["subcategory"].tolist()):
        ref_orf=gene[1][gene[1]["subcategory"]=="reference_match"]["ORF_seq"].unique()[0]
        for iso in gene[1].groupby(by="isoform", dropna=False):
            # no 5' degradation
            if iso[1]["subcategory"].values[0] not in possible_5degradation:
                genes_wo_deg=pd.concat([genes_wo_deg,iso[1]])
            # 5'degradation with no effect on ORF
            elif iso[1]["ORF_seq"].values[0] == ref_orf and iso[1]["subcategory"].values[0] in possible_5degradation:
                iso[1]["subcategory"]=["reference_match"]
                genes_wo_deg=pd.concat([genes_wo_deg,iso[1]])
            else:
            # 5' degradation with effect of ORF
                if iso[1]["ORF_seq"].values[0].isna():
                    iso[1]["subcategory"]=["reference_match"]
                    iso[1]["ORF_seq"]=[ref_orf]
                    genes_wo_deg=pd.concat([genes_wo_deg,iso[1]])                    
                elif len(iso[1]["ORF_seq"].values[0]) > len(ref_orf):
                    genes_wo_deg=pd.concat([genes_wo_deg,iso[1]])
                else:
                    iso[1]["subcategory"]=["reference_match"]
                    iso[1]["ORF_seq"]=[ref_orf]
                    genes_wo_deg=pd.concat([genes_wo_deg,iso[1]])
    elif 'reference_match' in gene[1]["subcategory"].tolist() and any(x in possible_5degradation for x in gene[1]["subcategory"].tolist()) == False:
        # add genes with ref, but without 5' alteration
        genes_wo_deg=pd.concat([genes_wo_deg, gene[1]])
    elif 'reference_match' not in gene[1]["subcategory"].tolist() and any(x in possible_5degradation for x in gene[1]["subcategory"].tolist()) == False:
        # add genes with ref, but without 5' alteration
        genes_wo_deg=pd.concat([genes_wo_deg, gene[1]])
    else:
        genes_uk=pd.concat([genes_uk, gene[1]])

NameError: name 'at_hodge' is not defined

In [3]:
def remove_5deg(acc):
    # load sqanti classification file
    at=pd.read_csv(f"/tmp/global2/lvaness/dl20-annex/final/isoseq/sqanti/classifications/{acc}-hodgepodgemerged_classification.txt", sep="\t")
    at["gene"]=at["isoform"].str.split(".").str[0]
    at["has_deg"]=0
    
    # get associated hodgepodge gene (one per row)
    span = 2
    at["hodgepodge_gene"]=at["associated_gene"].transform(lambda x: ["_".join(x.split("_")[i:i+span]) for i in range(0, len(x.split("_")), span)])
    at_hodge=at.explode("hodgepodge_gene")
    at_hodge=at_hodge.loc[at_hodge["hodgepodge_gene"].str.split("_").str[0]==acc]
    
    # start analysis per gene
    possible_5degradation=["alternative_5end","3prime_fragment"]
    genes_wo_deg=pd.DataFrame()
    genes_uk=pd.DataFrame()
    genes_ref_na=pd.DataFrame()
    for gene in at_hodge.groupby(by="hodgepodge_gene", dropna=False):
        # handle genes with reference match an 5' alteration
        if 'reference_match' in gene[1]["subcategory"].tolist() and any(x in possible_5degradation for x in gene[1]["subcategory"].tolist()):
            ### TODO: ADD column which says if we found 5' alteration or not
            ref_orf=gene[1][gene[1]["subcategory"]=="reference_match"]["ORF_seq"].unique()[0]
            gene[1]["has_deg"]=[1 for i in range(len(gene[1]))]
            if isinstance(ref_orf, float):
                # MAKE STATISTIC PER ACCESSION
                genes_ref_na=pd.concat([genes_ref_na,gene[1]])
                continue
            else:
                for iso in gene[1].groupby(by="isoform"):
                    # no 5' degradation
                    if iso[1]["subcategory"].values[0] not in possible_5degradation:
                        genes_wo_deg=pd.concat([genes_wo_deg,iso[1]])
                    # 5'degradation with no effect on ORF
                    elif iso[1]["ORF_seq"].values[0] == ref_orf and iso[1]["subcategory"].values[0] in possible_5degradation:
                        iso[1]["subcategory"]=["reference_match"]
                        genes_wo_deg=pd.concat([genes_wo_deg,iso[1]])
                    else:
                        # TODO: separate ORFs longer from ORFs shorter
                        try:
                            if isinstance(iso[1]["ORF_seq"].values[0], float):
                                iso[1]["subcategory"]=["reference_match"]
                                iso[1]["ORF_seq"]=[ref_orf]
                                genes_wo_deg=pd.concat([genes_wo_deg,iso[1]])
                            elif len(iso[1]["ORF_seq"].values[0]) > len(ref_orf):
                                genes_wo_deg=pd.concat([genes_wo_deg,iso[1]])
                            else:
                                iso[1]["subcategory"]=["reference_match"]
                                iso[1]["ORF_seq"]=[ref_orf]
                                genes_wo_deg=pd.concat([genes_wo_deg,iso[1]])
                        except:
                            print(iso[1]["ORF_seq"].values[0] + "= iso_orf")
        elif 'reference_match' in gene[1]["subcategory"].tolist() and any(x in possible_5degradation for x in gene[1]["subcategory"].tolist()) == False:
            # add genes with ref, but without 5' alteration
            genes_wo_deg=pd.concat([genes_wo_deg, gene[1]])
        elif 'reference_match' not in gene[1]["subcategory"].tolist() and any(x in possible_5degradation for x in gene[1]["subcategory"].tolist()) == False:
            # add genes with ref, but without 5' alteration
            genes_wo_deg=pd.concat([genes_wo_deg, gene[1]])
        else:
            genes_uk=pd.concat([genes_uk, gene[1]])
    genes_wo_deg["deg_removed"]=1
    genes_uk["deg_removed"]=0
    genes_ref_na["deg_removed"]=2
    all_genes=pd.concat([genes_wo_deg, genes_uk, genes_ref_na])

    all_genes.to_csv(f"output/09/{acc}-hodgepodgemerged_classification_wo5deg.txt", sep="\t", index=False)

In [4]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

if __name__ == '__main__':
    with Pool(17) as p:
        print(p.map(remove_5deg, accessions))

[None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None]


### Subset Sqanti

In [5]:
# This is the code for a subsetted sqanti table
accessions= ['at6923','at6929','at7143','at8285','at9104','at9336','at9503','at9578','at9744','at9762','at9806','at9830','at9847','at9852','at9879','at9883','at9900']
col=['isoform', 'chrom', 'strand', 'length', 'exons', 'structural_category',
       'associated_gene', 'associated_transcript', 'ref_length', 'ref_exons',
       'diff_to_TSS', 'diff_to_TTS', 'diff_to_gene_TSS', 'diff_to_gene_TTS',
       'subcategory', 'RTS_stage', 'all_canonical',
       'bite', 'FSM_class','ORF_seq','deg_removed']
span = 2

for acc in tqdm(accessions):

    # prepare sqanti output
    sqanti = pd.read_csv(f"output/09/{acc}-hodgepodgemerged_classification_wo5deg.txt", sep="\t",dtype={'ref_length': 'Int64', 'ref_exons': 'Int64','diff_to_TSS': 'Int64', 'diff_to_TTS': 'Int64',
                              'diff_to_gene_TSS': 'Int64', 'diff_to_gene_TTS': 'Int64'})
    #sqanti.fillna("NaN", inplace=True)
    sqanti = sqanti[col]
    
    # prepare read support
    reads = pd.read_csv(f"/ebio/abt6_projects7/dliso/dlis/leon_pipeline/output/v2/collapse_tama/{acc}.collapsed.tama.v2_read_support.txt", sep="\t")
    
    # also add gene read count?
    #r_sub = reads[["merge_trans_id","trans_read_count","gene_read_count"]]
    #r_sub.columns = ["isoform", "isoform_reads", "gene_reads"]
    r_sub = reads[["merge_trans_id","trans_read_count"]]
    r_sub.columns = ["isoform", "isoform_reads"]
    
    # add accession ID and merge all information
    acc_df = pd.merge(left=sqanti, right=r_sub, how='left', left_on='isoform', right_on='isoform')
    acc_df["hodgepodge_gene"]=acc_df["associated_gene"].transform(lambda x: ["_".join(x.split("_")[i:i+span]) for i in range(0, len(x.split("_")), span)])
    acc_df=acc_df.explode("hodgepodge_gene")
    acc_df.sort_values(by="hodgepodge_gene", inplace=True)
    
    ### prepare gene type (TODOc)
    nlr = pd.read_csv(f"/ebio/abt6_projects7/dliso/dlis/leon_pipeline/difflines_SQANTI/input/ref_sets/new_ids/nlr_{acc}.txt", sep="\t", names=["hodgepodge_gene"])
    nlr["type"]="nlr"
    busco = pd.read_csv(f"/ebio/abt6_projects7/dliso/dlis/leon_pipeline/difflines_SQANTI/input/ref_sets/new_ids/busco_{acc}.txt", sep="\t", names=["hodgepodge_gene"])
    busco["type"]="busco"
    house=pd.read_csv(f"/ebio/abt6_projects7/dliso/dlis/leon_pipeline/difflines_SQANTI/input/ref_sets/new_ids/housekeeping_{acc}.txt", sep="\t", names=["hodgepodge_gene"])
    house["type"]="house"
    all_sets=pd.concat([nlr, busco, house])
    all_sets
    
    # add gene type
    merged_all=pd.merge(left=acc_df, right=all_sets, how='left', left_on='hodgepodge_gene', right_on='hodgepodge_gene')
    
    merged_all['accession'] = acc
    merged_all=merged_all.loc[merged_all["hodgepodge_gene"].str.split("_").str[0]==acc]
    for c in merged_all.columns:
        if c in ['ref_length', 'ref_exons','diff_to_TSS', 'diff_to_TTS', 'diff_to_gene_TSS', 'diff_to_gene_TTS','ORF_seq']:
            merged_all[c] = merged_all[c].astype(str).replace("<NA>","NaN")
    merged_all["type"].fillna("other", inplace=True)
    
    merged_all.sort_values("hodgepodge_gene")

    # merged_all.to_csv(f"/tmp/global2/lvaness/dl20-annex/final_isoseq/sqanti/classification/{acc}-hodgepodgemerged_classification_hodgepodge-subset.txt", sep="\t", index=False)
    merged_all.to_csv(f"output/09/{acc}-hodgepodgemerged_classification_wo5deg-subset.txt", sep="\t", index=False)

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 17/17 [02:21<00:00,  8.34s/it]


### Calculate Diversity

In [9]:
def simpson(frequencies):
    return sum(f**2 for f in frequencies)

In [10]:
import warnings
warnings.simplefilter(action='ignore', category=pd.errors.PerformanceWarning)

#for acc in tqdm(accessions):
def get_entropies_per_gene(acc):
    # load sqanti classification file
    at=pd.read_csv(f"output/09/{acc}-hodgepodgemerged_classification_wo5deg-subset.txt", sep="\t")
    at["cat_subcat"]=at["structural_category"] + "_" + at["subcategory"]
    
    # get associated hodgepodge gene (one per row)
    #span = 2
    #at["hodgepodge_gene"]=at["associated_gene"].transform(lambda x: ["_".join(x.split("_")[i:i+span]) for i in range(0, len(x.split("_")), span)])
    #at_hodge=at.explode("hodgepodge_gene")
    #at_hodge=at_hodge.loc[at_hodge["hodgepodge_gene"].str.split("_").str[0]==acc]
    
    at_hodge=at


    # subset for analysis
    at_info=at_hodge[["chrom","strand","cat_subcat","isoform_reads","hodgepodge_gene", "isoform","ORF_seq","deg_removed"]]
    
    # start analysis per gene
    acc_df=pd.DataFrame()
    for gene in at_info.groupby(by="hodgepodge_gene"):
        #print(gene[0])
        #print(gene[1])
        gene[1].fillna("noncoding", inplace=True)
        #print(gene[0])
        g_df=pd.DataFrame()
        #get gene and strand
        g_df["hodgepodge_gene"]=[gene[0]]
        g_df["strand"]=gene[1]["strand"].unique()[0]
        g_df["chrom"]=gene[1]["chrom"].unique()[0]
        g_df["deg_removed"]=gene[1]["deg_removed"].unique()[0]
        #prepare analyses
        gene_rs=gene[1]["isoform_reads"].sum()
        gene_isoforms=len(gene[1]["isoform"])
        
        g_df["read_support"]=gene_rs
        g_df["isoforms"]=gene_isoforms

        #calculate diversity based on isoforms and their read supports
        isoform_frequencies=[]
        for i in gene[1]["isoform"].unique().tolist():
            #get read support for cat-combination
            rs=gene[1].loc[gene[1]["isoform"]==i]["isoform_reads"].sum() 
            isoform_frequencies.append(rs/gene_rs)
        
        g_df["isoform_shannon"]= entropy(isoform_frequencies)
        g_df["isoform_simpson"]= simpson(isoform_frequencies)
        
        
        ORF_frequencies=[]
        #calculate diversity based on reads, supporting ORFs
        for i in gene[1]["ORF_seq"].unique().tolist():
            #get read support for cat-combination
            rs=gene[1].loc[gene[1]["ORF_seq"]==i]["isoform_reads"].sum() 
            ORF_frequencies.append(rs/gene_rs)
            
        g_df["orf_shannon"]=entropy(ORF_frequencies)
        g_df["orf_simpson"]=simpson(ORF_frequencies)

        #calculate shannon entropy based on isoforms, supporting cat-subcat combinations
        #shannon_iso_cat_subcat=[]
        #for i in gene[1]["cat_subcat"].unique().tolist():
        #    #get n idoforms of for cat-combination
        #    n_iso=len(gene[1].loc[gene[1]["cat_subcat"]==i])  
        #    shannon_iso_cat_subcat.append(n_iso/gene_isoforms)
        #g_df["cat-subcat_iso_shannon"]=entropy(shannon_iso_cat_subcat)

        #calculate shannon entropy based on reads, supporting cat-subcat combinations
        #shannon_rs_cat_subcat=[]
        #for i in gene[1]["cat_subcat"].unique().tolist():
        #    #get read support for cat-combination
        #    rs=gene[1].loc[gene[1]["cat_subcat"]==i]["isoform_reads"].sum() 
        #    shannon_rs_cat_subcat.append(rs/gene_rs)
        #g_df["cat-subcat_rs_shannon"]=entropy(shannon_rs_cat_subcat)
        
        #shannon_iso_orf=[]
        #calculate shannon entropy based on isoforms, supporting ORFs
        #shanon_iso_cat_orf=[]
        #for i in gene[1]["ORF_seq"].unique().tolist():
        #    n_orf=len(gene[1].loc[gene[1]["ORF_seq"]==i])  
        #    shannon_iso_orf.append(n_orf/gene_isoforms)
        #g_df["orf_iso_shannon"]=entropy(shannon_iso_orf)
        
        acc_df = pd.concat([acc_df, g_df])

        
    # prepare hodgepodge gff
    columns=["chrom","source","feature","hodgepodge_genomic_start_coord","hodgepodge_genomic_end_coord","score","strand","frame","attribute"]
    gff=pd.read_csv(f"/tmp/global2/lvaness/dl20-annex/final/annotation/latest/{acc}_hodgepodgemerged.gff3", sep="\t", header=None, names=columns)
    gff_genes=gff.loc[(gff["feature"]=="gene")|(gff["feature"]=="pseudogene")|(gff["feature"]=="pseudogenic_region")|(gff["feature"]=="transposable_element_gene")]
    gff_genes["hodgepodge_gene"]=gff_genes["attribute"].str.split(";").str[0].str.split("=").str[1]
    gff_genes_sub=gff_genes[["hodgepodge_gene","hodgepodge_genomic_start_coord","hodgepodge_genomic_end_coord"]]

    # add gene coordinates
    acc_df=pd.merge(left=acc_df, right=gff_genes_sub, how='left', left_on='hodgepodge_gene', right_on='hodgepodge_gene')        
        

    # prepare gene type
    nlr = pd.read_csv(f"/ebio/abt6_projects7/dliso/dlis/leon_pipeline/difflines_SQANTI/input/ref_sets/new_ids/nlr_{acc}.txt", sep="\t", names=["hodgepodge_gene"])
    nlr["type"]="nlr"
    busco = pd.read_csv(f"/ebio/abt6_projects7/dliso/dlis/leon_pipeline/difflines_SQANTI/input/ref_sets/new_ids/busco_{acc}.txt", sep="\t", names=["hodgepodge_gene"])
    busco["type"]="busco"
    house=pd.read_csv(f"/ebio/abt6_projects7/dliso/dlis/leon_pipeline/difflines_SQANTI/input/ref_sets/new_ids/housekeeping_{acc}.txt", sep="\t", names=["hodgepodge_gene"])
    house["type"]="house"
    all_sets=pd.concat([nlr, busco, house])
    
    custom_dict = {'nlr':0, 'house':1, 'busco':2}  
    all_sets['rank'] = all_sets['type'].map(custom_dict)
    all_sets=all_sets.sort_values(by="rank").drop_duplicates(subset=["hodgepodge_gene"])
    
    # add gene type
    acc_df=pd.merge(left=acc_df, right=all_sets, how='left', left_on='hodgepodge_gene', right_on='hodgepodge_gene')
    acc_df["type"].fillna("other", inplace=True)
    

    col=["chrom","hodgepodge_genomic_start_coord","hodgepodge_genomic_end_coord","strand","hodgepodge_gene","type"]
    for i in acc_df.columns:
        if i not in col:
            col.append(i)
    acc_df=acc_df[col]
    acc_df.fillna(0, inplace=True)
    
    if not os.path.isdir(f"output/08/{acc}"):
        os.makedirs(f"output/08/{acc}")
    
    acc_df["hodgepodge_genomic_start_coord"]=acc_df["hodgepodge_genomic_start_coord"].astype(int)
    acc_df["hodgepodge_genomic_end_coord"]=acc_df["hodgepodge_genomic_end_coord"].astype(int)
    
    acc_df.to_csv(f"output/09/{acc}-hodgepodgemerged_wo5deg_shannon_entropies-per-gene.bed", sep="\t", index=False)

In [11]:
if __name__ == '__main__':
    with Pool(17) as p:
        print(p.map(get_entropies_per_gene, accessions))

/tmp/ipykernel_99803/1440115202.py:94: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  gff_genes["hodgepodge_gene"]=gff_genes["attribute"].str.split(";").str[0].str.split("=").str[1]
/tmp/ipykernel_99803/1440115202.py:94: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  gff_genes["hodgepodge_gene"]=gff_genes["attribute"].str.split(";").str[0].str.split("=").str[1]
/tmp/ipykernel_99803/1440115202.py:94: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_

[None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None]


In [92]:
acc_df

,isoform,chrom,strand,length,exons,structural_category,associated_gene,associated_transcript,ref_length,ref_exons,...,diff_to_gene_TTS,subcategory,RTS_stage,all_canonical,bite,FSM_class,ORF_seq,deg_removed,isoform_reads,hodgepodge_gene
293,G1003.1,at9900_1_chr1,-,540,1,antisense,novelGene_at9900_G015210_AS,novel,<NA>,<NA>,...,<NA>,mono-exon,False,NaN,NaN,A,NaN,1,1,G015210_AS
3350,G1029.1,at9900_1_chr1,-,1903,1,antisense,novelGene_at9900_G015520_AS,novel,<NA>,<NA>,...,<NA>,mono-exon,False,NaN,NaN,A,NaN,1,1,G015520_AS
110,G10009.1,at9900_1_chr3,+,1787,1,antisense,novelGene_at9900_G204800_AS,novel,<NA>,<NA>,...,<NA>,mono-exon,False,NaN,NaN,A,NaN,1,1,G204800_AS
305,G10033.1,at9900_1_chr3,-,1590,7,antisense,novelGene_at9900_G205260_AS,novel,<NA>,<NA>,...,<NA>,multi-exon,True,non_canonical,False,A,MPLQMAPIPCSLTPNLRFLSSGVAFWKSPKAFSNVMLDDARSALPP...,1,1,G205260_AS
1420,G10144.1,at9900_1_chr3,+,4619,1,antisense,novelGene_at9900_G207170_AS_novelGene_at9900_G...,novel,<NA>,<NA>,...,<NA>,mono-exon,False,NaN,NaN,A,MNLGSISQCSHLLHVFGIDGFGLVSALLQVIEGLSVGELQLQLAII...,1,1,G207170_AS
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2533,G10233.43,at9900_1_chr4,+,3971,2,antisense,novelGene_at9900_G208390_AS,novel,<NA>,<NA>,...,<NA>,multi-exon,False,canonical,True,B,NaN,1,1,novelGene_at9900
2121,G10219.6,at9900_1_chr3,+,1830,1,antisense,novelGene_at9900_G208260_AS,novel,<NA>,<NA>,...,<NA>,mono-exon,False,NaN,NaN,B,NaN,1,1,novelGene_at9900
2122,G10219.7,at9900_1_chr3,+,2373,1,antisense,novelGene_at9900_G208260_AS,novel,<NA>,<NA>,...,<NA>,mono-exon,False,NaN,NaN,B,MILEALAGSKFWILESTILVASYTSMETHSLLDETLSLSLSLNLSL...,1,1,novelGene_at9900
1914,G10198.2,at9900_1_chr3,+,1641,1,antisense,novelGene_at9900_G208020_AS,novel,<NA>,<NA>,...,<NA>,mono-exon,False,NaN,NaN,B,NaN,1,1,novelGene_at9900
